# EDS Factor Model - Data Exploration Notebook

This notebook allows you to explore and manipulate data from the Snowflake table `EDS_FACTORS_FUNDAMENTALS_NTM_LTM`.

## Setup
- Connect to Snowflake
- Load data from the table
- Explore and manipulate the data interactively


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import data_retrieval
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for better viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [2]:
# Initialize Snowflake connection
retriever = data_retrieval.SnowflakeDataRetriever()
retriever.connect()
print("✓ Connected to Snowflake")

Successfully connected to Snowflake
✓ Connected to Snowflake


## Load Data from Snowflake

Retrieve data from the `EDS_FACTORS_FUNDAMENTALS_NTM_LTM` table.


In [3]:
# Load sample data (modify query as needed)
query = """
SELECT * 
FROM "EDS_DEV_BERKELEY"."BERKELEY"."EDS_FACTORS_FUNDAMENTALS_NTM_LTM"
WHERE DATE >= '2024-01-01'
LIMIT 10000
"""

df = retriever.execute_query(query)
print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
df.head()

Loaded 10000 rows
Columns: ['FACTSET_ID', 'EPS_NTM', 'EPS_LTM', 'SALES_NTM', 'SALES_LTM', 'EBITDA_NTM', 'EBITDA_LTM', 'COGS_NTM', 'COGS_LTM', 'DATE']


,FACTSET_ID,EPS_NTM,EPS_LTM,SALES_NTM,SALES_LTM,EBITDA_NTM,EBITDA_LTM,COGS_NTM,COGS_LTM,DATE
0,KW9D5G-R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-10-20
1,KW9D5G-R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-10-09
2,KW9D5G-R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-30
3,KW9D5G-R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-19
4,KW9D5G-R,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-10


## Basic Data Exploration


In [4]:
# Basic info about the dataframe
print("DataFrame Info:")
print(f"Shape: {df.shape}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMissing Values:")
print(df.isnull().sum())

DataFrame Info:
Shape: (10000, 10)

Data Types:
FACTSET_ID     object
EPS_NTM       float64
EPS_LTM       float64
SALES_NTM     float64
SALES_LTM     float64
EBITDA_NTM    float64
EBITDA_LTM    float64
COGS_NTM      float64
COGS_LTM      float64
DATE           object
dtype: object

Missing Values:
FACTSET_ID       0
EPS_NTM       7440
EPS_LTM       7125
SALES_NTM     7437
SALES_LTM     7087
EBITDA_NTM    7668
EBITDA_LTM    7658
COGS_NTM      8046
COGS_LTM      8010
DATE             0
dtype: int64


In [5]:
# Statistical summary
df.describe()

,EPS_NTM,EPS_LTM,SALES_NTM,SALES_LTM,EBITDA_NTM,EBITDA_LTM,COGS_NTM,COGS_LTM
count,2560.000000,2875.000000,2.563000e+03,2.913000e+03,2.332000e+03,2.342000e+03,1.954000e+03,1.990000e+03
mean,39.006706,38.680765,6.880853e+05,5.816695e+05,1.725642e+05,1.631159e+05,5.563329e+05,5.287409e+05
std,164.088754,140.768851,4.197265e+06,3.727526e+06,1.025412e+06,9.733222e+05,3.249046e+06,3.095025e+06
min,-16.344930,-249.000000,0.000000e+00,0.000000e+00,-3.412124e+02,-4.500000e+03,0.000000e+00,0.000000e+00
25%,0.083485,0.040701,4.497660e+02,4.116495e+02,6.653751e+01,4.721212e+01,2.355710e+02,2.345477e+02
50%,0.958153,0.739534,3.918728e+03,3.975908e+03,7.141399e+02,4.993821e+02,2.969979e+03,2.570647e+03
75%,3.488312,4.790601,1.009312e+04,1.003470e+04,2.155339e+03,2.223435e+03,7.267737e+03,7.117647e+03
max,1254.931500,1051.772600,3.230000e+07,2.853421e+07,7.869706e+06,6.964780e+06,2.156990e+07,2.038196e+07


In [9]:
# Convert DATE column to datetime if it's not already
if 'DATE' in df.columns:
    print(f"DATE column type before conversion: {df['DATE'].dtype}")
    print(f"Sample DATE value type: {type(df['DATE'].iloc[0])}")
    
    # Handle datetime.date objects from Snowflake by converting to string first
    # This ensures proper conversion to pandas datetime64
    if df['DATE'].dtype == 'object':
        # Convert date objects to datetime64 properly
        df['DATE'] = pd.to_datetime(df['DATE'].astype(str), errors='coerce')
    else:
        df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
    
    # Ensure it's datetime64[ns] type
    df['DATE'] = df['DATE'].astype('datetime64[ns]')
    
    print(f"DATE column type after conversion: {df['DATE'].dtype}")
    print(f"Sample DATE value type after: {type(df['DATE'].iloc[0])}")
    
    print(f"\nDate range: {df['DATE'].min()} to {df['DATE'].max()}")
    print(f"Number of unique dates: {df['DATE'].nunique()}")
    print(f"Number of null dates after conversion: {df['DATE'].isnull().sum()}")
    
# Check unique FACTSET_IDs
if 'FACTSET_ID' in df.columns:
    print(f"\nNumber of unique FACTSET_IDs: {df['FACTSET_ID'].nunique()}")
    print(f"\nSample FACTSET_IDs:")
    print(df['FACTSET_ID'].unique()[:10])

DATE column type before conversion: object
Sample DATE value type: <class 'datetime.date'>
DATE column type after conversion: datetime64[ns]
Sample DATE value type after: <class 'pandas._libs.tslibs.timestamps.Timestamp'>

Date range: 2024-01-01 00:00:00 to 2025-10-22 00:00:00
Number of unique dates: 473
Number of null dates after conversion: 0

Number of unique FACTSET_IDs: 148

Sample FACTSET_IDs:
['KW9D5G-R' 'HCZBFW-R' 'GJV9JP-R' 'H7KY9W-R' 'WSZS54-R' 'WSZV52-R'
 'V9BQ1X-R' 'L7Q82K-R' 'Q3MGYC-R' 'DKH143-R']


In [11]:
# Example: Filter by date range
# Ensure both sides are pandas datetime for comparison
filter_date = pd.to_datetime('2024-06-01')
df_filtered = df[df['DATE'] >= filter_date]
print(f"Filtered data: {len(df_filtered)} rows (from {len(df)} total)")
df_filtered.tail()

# Example: Group by FACTSET_ID and calculate statistics
# df_grouped = df.groupby('FACTSET_ID').agg({
#     'EPS_NTM': 'mean',
#     'EPS_LTM': 'mean',
#     'SALES_NTM': 'sum'
# }).reset_index()
# df_grouped.head()

# Example: Create derived columns
# df['EPS_GROWTH'] = (df['EPS_NTM'] - df['EPS_LTM']) / df['EPS_LTM']
# df['SALES_GROWTH'] = (df['SALES_NTM'] - df['SALES_LTM']) / df['SALES_LTM']

# Uncomment and modify the examples above, or write your own code here:


Filtered data: 7675 rows (from 10000 total)


,FACTSET_ID,EPS_NTM,EPS_LTM,SALES_NTM,SALES_LTM,EBITDA_NTM,EBITDA_LTM,COGS_NTM,COGS_LTM,DATE
9980,SKV1WK-R,0.056928,0.048427,331.38907,296.87445,33.295680,26.247871,169.80794,152.66576,2024-07-16
9981,SKV1WK-R,0.056618,0.048373,330.19280,296.42783,33.052310,26.153234,169.17708,152.58360,2024-07-05
9982,SKV1WK-R,0.056280,0.048266,329.18887,296.06230,32.844980,26.070482,168.66092,152.51639,2024-06-26
9983,SKV1WK-R,0.056111,0.048283,328.21097,295.69690,32.645890,25.993229,168.14474,152.44917,2024-06-17
9984,SKV1WK-R,0.055801,0.048229,327.01572,295.25027,32.402565,25.898808,167.51389,152.36703,2024-06-06


In [13]:
df_filtered['EBITDA_NTM'] = df_filtered['EBITDA_NTM'].astype(float)
df_filtered['EBITDA_LTM'] = df_filtered['EBITDA_LTM'].astype(float)
df_filtered['EBITDA_GROWTH'] = (df_filtered['EBITDA_NTM'] - df_filtered['EBITDA_LTM']) / df_filtered['EBITDA_LTM']
df_filtered[df_filtered['EBITDA_NTM'] < 0]

/var/folders/4_/39blryb167zcxjxrb_2zv0b40000gn/T/ipykernel_7467/1356917326.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['EBITDA_NTM'] = df_filtered['EBITDA_NTM'].astype(float)
/var/folders/4_/39blryb167zcxjxrb_2zv0b40000gn/T/ipykernel_7467/1356917326.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['EBITDA_LTM'] = df_filtered['EBITDA_LTM'].astype(float)
/var/folders/4_/39blryb167zcxjxrb_2zv0b40000gn/T/ipykernel_7467/1356917326.py:3: SettingWithCopyWarning: 
A value is tryin

,FACTSET_ID,EPS_NTM,EPS_LTM,SALES_NTM,SALES_LTM,EBITDA_NTM,EBITDA_LTM,COGS_NTM,COGS_LTM,DATE,EBITDA_GROWTH
541,Q3MGYC-R,-0.131781,NaN,0.0,NaN,-12.418836,NaN,NaN,NaN,2025-10-17,NaN
542,Q3MGYC-R,-0.130795,NaN,0.0,NaN,-12.229095,NaN,NaN,NaN,2025-10-08,NaN
543,Q3MGYC-R,-0.129808,NaN,0.0,NaN,-12.039356,NaN,NaN,NaN,2025-09-29,NaN
544,Q3MGYC-R,-0.121452,NaN,NaN,NaN,-12.868603,NaN,NaN,NaN,2025-09-18,NaN
545,Q3MGYC-R,-0.120712,NaN,NaN,NaN,-12.597616,NaN,NaN,NaN,2025-09-09,NaN
...,...,...,...,...,...,...,...,...,...,...,...
6960,GY8Y0L-R,-0.161342,NaN,NaN,NaN,-30.727398,NaN,NaN,NaN,2025-10-13,NaN
6961,GY8Y0L-R,-0.160137,NaN,NaN,NaN,-30.410960,NaN,NaN,NaN,2025-10-02,NaN
6962,GY8Y0L-R,-0.159151,NaN,NaN,NaN,-30.152056,NaN,NaN,NaN,2025-09-23,NaN
6963,GY8Y0L-R,-0.157945,NaN,NaN,NaN,-29.835617,NaN,NaN,NaN,2025-09-12,NaN


## Visualization Examples


In [ ]:
# Example: Plot time series for a specific FACTSET_ID
# Uncomment and modify as needed:
# 
# if 'FACTSET_ID' in df.columns and 'DATE' in df.columns:
#     # Pick a specific company
#     company_id = df['FACTSET_ID'].unique()[0]
#     df_company = df[df['FACTSET_ID'] == company_id].sort_values('DATE')
#     
#     plt.figure(figsize=(12, 6))
#     plt.plot(df_company['DATE'], df_company['EPS_NTM'], label='EPS NTM')
#     plt.plot(df_company['DATE'], df_company['EPS_LTM'], label='EPS LTM')
#     plt.xlabel('Date')
#     plt.ylabel('EPS')
#     plt.title(f'EPS Over Time - {company_id}')
#     plt.legend()
#     plt.xticks(rotation=45)
#     plt.tight_layout()
#     plt.show()


In [ ]:
# Example: Distribution plots
# Uncomment to use:
# 
# numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# if len(numeric_cols) > 0:
#     fig, axes = plt.subplots(2, 3, figsize=(15, 10))
#     axes = axes.ravel()
#     
#     for idx, col in enumerate(numeric_cols[:6]):
#         df[col].hist(bins=50, ax=axes[idx])
#         axes[idx].set_title(col)
#     
#     plt.tight_layout()
#     plt.show()


## Custom Queries

Create custom SQL queries to load specific data subsets.


In [ ]:
# Custom query example - modify as needed
# 
# custom_query = """
# SELECT 
#     FACTSET_ID,
#     DATE,
#     EPS_NTM,
#     EPS_LTM,
#     SALES_NTM
# FROM "EDS_DEV_BERKELEY"."BERKELEY"."EDS_FACTORS_FUNDAMENTALS_NTM_LTM"
# WHERE FACTSET_ID = 'X1SDWS-R'
#   AND DATE >= '2024-01-01'
# ORDER BY DATE DESC
# """
# 
# df_custom = retriever.execute_query(custom_query)
# df_custom.head()


## Save Processed Data

Save your processed dataframe to a file.


In [ ]:
# Example: Save to CSV
# df.to_csv('processed_data.csv', index=False)
# print("Data saved to processed_data.csv")

# Example: Save to Excel
# df.to_excel('processed_data.xlsx', index=False)
# print("Data saved to processed_data.xlsx")

# Example: Save to pickle (preserves data types)
# df.to_pickle('processed_data.pkl')
# print("Data saved to processed_data.pkl")


In [ ]:
# Clean up: Close Snowflake connection
retriever.disconnect()
print("✓ Disconnected from Snowflake")
